In [1]:
import sys
sys.path.append('..')

In [ ]:
from forecast.data import *
from forecast.model import *
from config import *
import matplotlib.pyplot as plt
import torch.nn as nn
from tqdm import tqdm
import hvplot
import hvplot.xarray

In [3]:
data_path = "../"+ config.SSTA_DAILY_PATH
landmask_path = "../"+ config.LANDMASK_PATH

In [4]:
ds_xr = xr.open_zarr(str(data_path))

In [5]:
ds_xr.ssta

<xarray.DataArray 'ssta' (time: 15566, lat: 180, lon: 360)> Size: 4GB
dask.array<open_dataset-ssta, shape=(15566, 180, 360), dtype=float32, chunksize=(365, 180, 360), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 125kB 1981-09-01 1981-09-02 ... 2026-04-14
  * lat        (lat) float32 720B -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
  * lon        (lon) float32 1kB 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
    dayofyear  (time) int64 125kB dask.array<chunksize=(365,), meta=np.ndarray>
Attributes:
    long_name:     Daily Sea Surface Temperature
    units:         degC
    valid_range:   [-3.0, 45.0]
    precision:     2.0
    dataset:       NOAA High-resolution Blended Analysis
    var_desc:      Sea Surface Temperature
    level_desc:    Surface
    statistic:     Mean
    parent_stat:   Individual Observations
    actual_range:  [-1.7999999523162842, 34.81999969482422]

In [15]:
test_ds = SSTADataset(data_path, landmask_path, time_range=config.DL_TEST_RANGE, n_out=14)
train_ds = SSTADataset(data_path, landmask_path, time_range=config.DL_TRAIN_RANGE, n_out=14)

x, y = test_ds[0]
print(x.shape, y.shape)

torch.Size([14, 90, 180]) torch.Size([14, 90, 180])


In [31]:
loader = DataLoader(test_ds, batch_size=1, shuffle=False,  drop_last=True)
x_ld, y_ld = next(iter(loader))
print(x_ld.shape, y_ld.shape)

torch.Size([1, 14, 90, 180]) torch.Size([1, 14, 90, 180])


In [8]:
lstm_model = ConvLSTMForecast(n_out=8)

In [9]:
out = lstm_model(x_ld)

In [12]:
from forecast.evaluate import reduce_factor, spatial_acc_batch
from forecast.baselines import persistence_forecast, RidgeBaseline
LEAD_TIMES = (1, 3, 7, 14)

In [17]:
ridge = RidgeBaseline(lead_times=LEAD_TIMES)
ridge.fit(train_ds, test_ds.land_mask)


  Ridge fitted on 690,880 (pixel×sample) pairs


### Forecasting

In [19]:
ocean = ~test_ds.land_mask   # (H, W) bool
ocean_t = torch.from_numpy(ocean)


In [148]:

methods = ["model", "persistence", "ridge"]
accum = {
    m: {k: {"sse": 0.0, "n": 0, "acc_sum": 0.0, "acc_n": 0}
        for k in LEAD_TIMES}
    for m in methods
}

spatial_preds  = {k: [] for k in LEAD_TIMES}
spatial_truths = {k: [] for k in LEAD_TIMES}


In [73]:
def pixel_acc_map(pred: np.ndarray, truth: np.ndarray, axis:int) -> np.ndarray:
    # pred, truth: (N, H, W)
    p = pred  - pred.mean(axis=axis, keepdims=True)
    t = truth - truth.mean(axis=axis, keepdims=True)
    num = (p * t).sum(axis=axis)
    den = np.sqrt((p**2).sum(axis=axis) * (t**2).sum(axis=axis)) + 1e-12
    return num / den   # (H, W)


In [156]:

methods = ["model", "persistence", "ridge"]
accum = {
    m: {k: {"sse": 0.0, "n": 0, "acc_sum": 0.0, "acc_n": 0}
        for k in LEAD_TIMES}
    for m in methods
}

spatial_preds  = {k: [] for k in LEAD_TIMES}
spatial_truths = {k: [] for k in LEAD_TIMES}


for x, y in tqdm(loader, desc="evaluating"):
    x = x
    y = y

    preds = {
        "ridge":       ridge.predict_all_leads(x.cpu(), LEAD_TIMES[-1]),
    }
    for m, pred in preds.items():
        for k in LEAD_TIMES:
            if k > pred.shape[1]:
                continue
            
            pred_k = pred[:, k - 1]
            target_k = y[:, k - 1]

            # set all land at 0 insteak of masking to preserve shape
            pred_k[:, test_ds.land_mask] = 0
            target_k[:, test_ds.land_mask] = 0

            spatial_preds[k].append(pred_k)   # (B, H, W)
            spatial_truths[k].append(target_k)

            a = accum[m][k]
            sse = float(((pred_k - target_k) ** 2).sum().item())
            print(sse)
            a["sse"] += sse
            a["n"]   += pred_k.numel()
        break

    break

            # p_np = pred_k.cpu().float().numpy()    # (B, n_ocean)
            # t_np = target_k.cpu().float().numpy()



evaluating:   0%|          | 0/1799 [00:00<?, ?it/s]

289.34393310546875
1128.567626953125
1988.4072265625
2949.3662109375


In [105]:
acc_all_map = []
for k in LEAD_TIMES:
    a = accum[m][k]
    # rmse = (a["sse"] / max(a["n"], 1)) ** 0.5

    all_pred  = np.concatenate(spatial_preds[k],  axis=0)  # (N, H, W)
    all_truth = np.concatenate(spatial_truths[k], axis=0)
    acc_map = pixel_acc_map(all_pred, all_truth, axis=0)            # (H, W)
    acc_map[ds.land_mask] = np.nan                             # mask land

    acc_all_map.append(acc_map)
    print(acc_map.shape)


(90, 180)
(90, 180)
(90, 180)
(90, 180)


In [109]:
acc_all_map = np.array(acc_all_map)

In [125]:
ds_xr = ds_xr.coarsen(lat=2, lon=2).mean()
ds_xr.ssta.shape

(15566, 90, 180)

In [137]:
ds = xr.Dataset(
    data_vars=dict(
        acc_all_map=(['lead_time', 'lat', 'lon'], acc_all_map),
    ),
    coords = dict(
        lead_time=np.array(LEAD_TIMES),
        lat=("lat", ds_xr.ssta.lat.values),
        lon=("lon", ds_xr.ssta.lon.values),
    ),
    attrs=dict(
        metric='Anomaly Correlation Coefficient',
        model='Ridge Regression')
)

In [135]:
ds.to_zarr(f'{ds.model}_{ds.metric}_')

<xarray.Dataset> Size: 260kB
Dimensions:      (lead_time: 4, lat: 90, lon: 180)
Coordinates:
  * lead_time    (lead_time) int64 32B 1 3 7 14
  * lat          (lat) float32 360B -89.0 -87.0 -85.0 -83.0 ... 85.0 87.0 89.0
  * lon          (lon) float32 720B 1.0 3.0 5.0 7.0 ... 353.0 355.0 357.0 359.0
Data variables:
    acc_all_map  (lead_time, lat, lon) float32 259kB nan nan ... 0.883 0.8776
Attributes:
    description:  Anomaly Correlation Coefficient of Sea Surface Temperature
    model:        Ridge Regression

In [136]:
# da = xr.DataArray(acc_map, dims=["lat", "lon"])
ds.hvplot(x='lon', y='lat', cmap="RdYlBu_r", clim=(-1, 1), title=f'Anomaly Correlation Coefficient at different Lead Times using {ds.model}')

BokehModel(combine_events=True, render_bundle={'docs_json': {'471273aa-ffd3-4c9b-96f6-0f2e80bb69c8': {'version…